# StratScout — lighter data: radar-only video + `.dem` positions

Companion to `stratscout_prototype.ipynb`. Self-contained; imports nothing local.

Rendering full matches is the slow, heavy part of the project: **~2.5 GB and ~40 minutes of
playback per match**. But only one model actually needs pixels:

| Needs | Detector (Model 1) | Classifier (Model 2), heatmaps, win rates |
|---|---|---|
| Input | radar **pixels** | player **positions** `(32, 10, 2)` |
| Source here | **Part A** — a few rendered matches, shrunk to radar-only | **Part B** — parsed straight from `.dem`, no rendering |

Like a driving school: you only need a real car for the practical lessons; the highway code can
be learned from the book. The detector is the practical lesson.

**The pixels-only rule still holds.** `.dem` files are used here during development, for
training data and labels. The deployed pipeline still sees only video:
pixels → detector → tracker → classifier. Training Model 2 on demo positions and testing it on
tracker positions from held-out rendered matches *is* Experiment A vs B — this notebook just
makes the A side cheap enough to scale.

**What not to do:** draw our own radar images from demo positions and train the detector on
them. That is the closed loop the main notebook's synthetic mode warns about.

### Outputs

| File | What |
|---|---|
| `data_radar/<match>.mp4` | radar-only clip, ~80 MB per match instead of ~2.5 GB |
| `radar_fit.json` | fine-tuning of the world → crop mapping |
| `labels/<demo>.demo_rounds.json` | your tactic labels per demo round |
| `dem_rounds.npz` | labelled rounds in the main notebook's `positions` format |

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import re
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import matplotlib.pyplot as plt
import cv2

try:
    from demoparser2 import DemoParser
except ImportError:
    DemoParser = None
    print("demoparser2 not installed — Part B is skipped.   pip install demoparser2")

CLASSES = ["A_RUSH", "B_RUSH", "A_SPLIT", "B_SPLIT", "RETAKE"]   # must match the main notebook
EXCLUDED = "EXCLUDED"     # a round that fits no class (slow default etc.) — see CLAUDE.md §9


@dataclass
class DemCfg:
    # --- Part A: radar-only video -------------------------------------------
    data_dir: str = "data"              # full-screen renders (same folder as the main notebook)
    radar_dir: str = "data_radar"       # radar-only output
    crops_json: str = "crops.json"      # crop box per video, written by the main notebook §3
    radar_fps: float = 8.0              # main notebook samples at <= 4 fps; 8 leaves headroom

    # --- Part B: demos --------------------------------------------------------
    demo_dir: str = "demos"             # .dem files, searched recursively (subfolders fine)
    map_name: str = "de_mirage"
    tickrate: int = 64
    seq_len: int = 32                   # same as the main notebook's CFG.seq_len
    n_per_team: int = 5

    # de_mirage overview: world units -> 1024 px overview image. Verified on your footage:
    # 99.1% of demo positions land on walkable radar pixels with no further adjustment.
    overview_pos_x: float = -3230.0
    overview_pos_y: float = 1713.0
    overview_scale: float = 5.0
    overview_px: int = 1024

    radar_fit_json: str = "radar_fit.json"
    labels_dir: str = "labels"
    dem_npz: str = "dem_rounds.npz"
    map_image: str = "mirage-radar.png"    # clean radar screenshot: background for every plot


CFG = DemCfg()

# measured anchors on the crop (CLAUDE.md §13) — used to name the plant site
SITE_ANCHORS = {"A": (0.545, 0.770), "B": (0.230, 0.276)}
TEAM_COLOUR = {"T": "#ff9020", "CT": "#40a0ff"}

## 2. Part A — shrink rendered matches to radar-only clips

The main notebook throws away everything outside the radar the moment it reads a frame, so
storing it is pure waste. Measured on your two matches:

| | full screen 1280×720 @ 60 fps | **radar only 192×192 @ 8 fps** | time |
|---|---|---|---|
| `aurora_vs_betboom.mp4` | 2,434 MB | **78 MB** | 68 s |
| `vitality_vs_falcons.mp4` | 2,560 MB | **83 MB** | 73 s |

About 31× smaller. Compression barely touches the icons: median hue shift 2 (the finder
tolerates 22), PSNR 38.9 dB. This OpenCV build has no H.264 encoder, so it falls back to
MPEG-4 Part 2 (the `openh264` warning is harmless); `ffmpeg` with H.264 would roughly halve it
again. Uses each video's crop box from `crops.json`, so run the main notebook's §1–3 first.

**Using the clips in the main notebook:** set `data_dir="data_radar"` and
`crop_box=(0, 0, 192, 192)` in its `Config`. A pre-cropped clip has no surrounding screen for
auto-crop to compare against; the fixed box skips that search. Nothing else changes, because
times are in seconds, not frame numbers.

Going forward you can skip the 2.5 GB step entirely: crop the game capture to the radar in OBS
and record at 8–30 fps. Keep the radar settings in CLAUDE.md §4 unchanged.

In [ ]:
def _open_writer(path: Path, fps: float, size: Tuple[int, int]) -> cv2.VideoWriter:
    """H.264 if this OpenCV build has it, else MPEG-4 Part 2 (still ~31x smaller)."""
    for fourcc in ("avc1", "mp4v"):
        vw = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*fourcc), fps, size)
        if vw.isOpened():
            return vw
    raise RuntimeError("OpenCV cannot write .mp4 on this machine")


def radar_only_clip(video: Path, box: Sequence[int], out: Path, fps: float = None) -> Path:
    """Full-screen render -> radar-only clip at `fps`. Decodes once, front to back."""
    fps = fps or CFG.radar_fps
    cap = cv2.VideoCapture(str(video))
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 60.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    x0, y0, x1, y1 = box
    out.parent.mkdir(parents=True, exist_ok=True)
    tmp = out.with_name("~" + out.name)        # '~' prefix: the main notebook ignores it
    vw = _open_writer(tmp, fps, (x1 - x0, y1 - y0))
    step, keep_at, written = src_fps / fps, 0.0, 0
    for i in range(total):
        if i < keep_at:
            if not cap.grab():                 # decode without converting: faster
                break
            continue
        ok, frame = cap.read()
        if not ok:
            break
        vw.write(np.ascontiguousarray(frame[y0:y1, x0:x1]))
        written += 1
        keep_at += step
    cap.release(); vw.release()
    tmp.replace(out)                           # only a finished clip gets the real name
    return out


def crop_for(video: Path, crops: Dict) -> Optional[List[int]]:
    """Crop box by file name, else by byte size: renaming a video does not change its size,
    so a renamed file still finds its box (crops.json is keyed by the old name)."""
    if video.name in crops:
        return crops[video.name]["crop_box"]
    size = video.stat().st_size
    hits = [c["crop_box"] for c in crops.values() if c.get("bytes") == size]
    return hits[0] if hits else None


def shrink_all(force: bool = False) -> None:
    crops_path = Path(CFG.crops_json)
    if not crops_path.exists():
        print(f"{crops_path} missing — run stratscout_prototype.ipynb §1-3 first"); return
    crops = json.loads(crops_path.read_text())
    videos = sorted(p for p in Path(CFG.data_dir).glob("*")
                    if p.suffix.lower() == ".mp4" and not p.name.startswith((".", "~")))
    for v in videos:
        out = Path(CFG.radar_dir) / f"{v.stem}.mp4"
        box = crop_for(v, crops)
        if box is None:
            print(f"{v.name:<36} no crop box in {crops_path.name} — run the main notebook §3"); continue
        if out.exists() and out.stat().st_mtime > v.stat().st_mtime and not force:
            print(f"{v.name:<36} already done ({out.stat().st_size / 1e6:.0f} MB)"); continue
        t = time.time()
        radar_only_clip(v, box, out)
        print(f"{v.name:<36} {v.stat().st_size / 1e6:7.0f} MB -> {out.stat().st_size / 1e6:5.1f} MB"
              f"   [{time.time() - t:.0f}s]")


shrink_all()

### 2.1 Radar images: one to measure, one to look at

* **`BG`, the median of many frames:** the map with the players washed out, like a
  long-exposure photo of a busy street. Used only by the §4 fit, to tell walkable radar pixels
  from black. It is slightly blurred and is never drawn.
* **`PLOT_BG`, `mirage-radar.png`:** your clean radar screenshot, the background for every plot.
  It frames the same area as the crop (correlation 0.986 with the median on a plain resize).

In [ ]:
def radar_background(n: int = 120) -> Optional[np.ndarray]:
    """RGB median radar from the first radar-only clip, else from a full render + its crop."""
    clips = sorted(Path(CFG.radar_dir).glob("*.mp4"))
    crops = json.loads(Path(CFG.crops_json).read_text()) if Path(CFG.crops_json).exists() else {}
    if clips:
        src, box = clips[0], None
    else:
        vids = [p for p in sorted(Path(CFG.data_dir).glob("*.mp4")) if crop_for(p, crops)]
        if not vids:
            return None
        src, box = vids[0], crop_for(vids[0], crops)
    cap = cv2.VideoCapture(str(src))
    total = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    frames = []
    for i in np.linspace(total * 0.05, total * 0.9, n).astype(int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, f = cap.read()
        if ok:
            f = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
            frames.append(f if box is None else f[box[1]:box[3], box[0]:box[2]])
    cap.release()
    return np.median(np.stack(frames), axis=0).astype(np.uint8) if frames else None


def load_plot_bg() -> Optional[np.ndarray]:
    raw = cv2.imread(CFG.map_image, cv2.IMREAD_UNCHANGED)
    if raw is None:
        return None
    if raw.ndim == 3 and raw.shape[2] == 4:                   # composite alpha over the dark panel
        a = raw[..., 3:4].astype(np.float32) / 255
        rgb = cv2.cvtColor(raw[..., :3], cv2.COLOR_BGR2RGB).astype(np.float32)
        return (rgb * a + np.array([12, 16, 24], np.float32) * (1 - a)).astype(np.uint8)
    return cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)


BG = radar_background()                                       # for measuring (§4 fit)
PLOT_BG = load_plot_bg()
if PLOT_BG is None:
    PLOT_BG = BG
    print(f"{CFG.map_image} not found — plots use the footage median")
print("fit reference:", "footage median" if BG is not None else "none (no footage yet)",
      "| plot background:", CFG.map_image if PLOT_BG is not None and PLOT_BG is not BG
      else ("footage median" if BG is not None else "dark panel"))

## 3. Part B — positions straight from `.dem`

`demoparser2` reads a demo in seconds and gives, per tick: every player's world position, team
and whether they are alive, plus round start (`round_freeze_end`), round end with the
**winner** and the reason, and bomb plants. So, per round, with no rendering:

* **positions** `(32, 10, 2)`, same contract as the tracker: slots 0–4 = T, 5–9 = CT,
  normalised to the crop. Dead players are `NaN` from death onwards, which is the honest
  version of what the tracker should do (CLAUDE.md §10, dead-player gap).
* **the winner**, so Objective 3 needs no keypress;
* **the plant site**, a useful hint when labelling.

Unlike the tracker, a slot here is always the same player (sorted by Steam ID), so this is
true oracle identity.

The window is `freeze_end → round_end`, sampled evenly to 32 steps like the main notebook.
That is **not** the same as a human-marked window in `RoundLabeller`. Keep the difference in
mind when comparing Experiment A (demo-trained) with B (video).

Drop `.dem` files into `demos/` (subfolders are fine). Demos for other maps are skipped.

In [ ]:
def list_demos(demo_dir: str = None) -> List[Path]:
    d = Path(demo_dir or CFG.demo_dir)
    d.mkdir(parents=True, exist_ok=True)
    return sorted(p for p in d.rglob("*") if p.is_file() and p.suffix.lower() == ".dem")


def match_group(stem: str) -> str:
    """'vitality-vs-falcons-m3-mirage' and 'vitality_vs_falcons' -> 'vitality_vs_falcons'.
    Used to pair a demo with its rendered video, which must land in the same split."""
    s = re.sub(r"[\s\-]+", "_", stem.lower())
    s = re.sub(r"_(m\d+_)?(de_)?mirage$", "", s)
    return re.sub(r"_m\d+$", "", s)


def _event(parser, name: str):
    import pandas as pd
    try:
        df = parser.parse_event(name)
        return df if isinstance(df, pd.DataFrame) and len(df) else pd.DataFrame(columns=["tick"])
    except Exception:
        return pd.DataFrame(columns=["tick"])


def _winner(w) -> str:
    return {"2": "T", "3": "CT"}.get(str(w), str(w))       # string in current demoparser2


def round_windows(parser) -> List[Dict]:
    """[{start_tick, end_tick, winner, reason}] for every completed round after match start."""
    starts = np.sort(_event(parser, "round_freeze_end")["tick"].to_numpy(dtype=np.int64))
    ends = _event(parser, "round_end").sort_values("tick")
    live = _event(parser, "round_announce_match_start")["tick"]
    if len(live):
        starts = starts[starts >= int(live.max())]          # drop warmup / restarts
    out = []
    for k, a in enumerate(starts):
        nxt = starts[k + 1] if k + 1 < len(starts) else np.iinfo(np.int64).max
        e = ends[(ends["tick"] > a) & (ends["tick"] <= nxt)]
        if len(e) == 0:
            continue                                         # round restarted / unfinished
        r = e.iloc[0]
        out.append({"start_tick": int(a), "end_tick": int(r["tick"]),
                    "winner": _winner(r.get("winner")), "reason": str(r.get("reason", ""))})
    return out


def parse_demo(path: Path) -> List[Dict]:
    """One demo -> list of round dicts, positions in WORLD units (mapped to the crop later)."""
    parser = DemoParser(str(path))
    header = parser.parse_header()
    if header.get("map_name") != CFG.map_name:
        print(f"{path.name}: map {header.get('map_name')} — skipped"); return []
    wins = round_windows(parser)
    plants = _event(parser, "bomb_planted")
    sample = [np.linspace(w["start_tick"], w["end_tick"], CFG.seq_len).round().astype(int)
              for w in wins]
    plant_ticks = [int(t) for t in plants["tick"]] if len(plants) else []
    ticks = sorted(set(np.concatenate(sample).tolist()) | set(plant_ticks))
    df = parser.parse_ticks(["X", "Y", "team_num", "is_alive"], ticks=ticks)
    df = df[df["team_num"].isin([2, 3])]
    by_tick = {t: g for t, g in df.groupby("tick")}

    rounds = []
    for ri, (w, st) in enumerate(zip(wins, sample)):
        first = by_tick.get(int(st[0]))
        if first is None:
            continue
        slots = []                                           # 5 T steamids, then 5 CT
        for team in (2, 3):
            ids = sorted(first.loc[first["team_num"] == team, "steamid"].tolist())[:CFG.n_per_team]
            slots += ids + [None] * (CFG.n_per_team - len(ids))
        world = np.full((CFG.seq_len, 2 * CFG.n_per_team, 2), np.nan, np.float32)
        for j, t in enumerate(st):
            g = by_tick.get(int(t))
            if g is None:
                continue
            g = g.set_index("steamid")
            for s, sid in enumerate(slots):
                if sid is not None and sid in g.index and bool(g.at[sid, "is_alive"]):
                    world[j, s] = (g.at[sid, "X"], g.at[sid, "Y"])
        plant_tick, plant_xy = None, None
        for _, p in plants.iterrows():
            if w["start_tick"] <= p["tick"] <= w["end_tick"]:
                g = by_tick.get(int(p["tick"]))
                if g is not None:
                    row = g[g["steamid"].astype(str) == str(p.get("user_steamid"))]
                    if len(row):
                        plant_tick = int(p["tick"])
                        plant_xy = (float(row["X"].iloc[0]), float(row["Y"].iloc[0]))
                break
        rounds.append({"demo": path.stem, "match_group": match_group(path.stem),
                       "round_idx": ri + 1, **w,
                       "duration_s": (w["end_tick"] - w["start_tick"]) / CFG.tickrate,
                       "plant_tick": plant_tick, "plant_world": plant_xy, "world": world})
    return rounds


DEMOS = list_demos()
print(f"{CFG.demo_dir}/ -> {len(DEMOS)} demo(s)")
DEM_ROUNDS: List[Dict] = []
if DemoParser is not None:
    for d in DEMOS:
        t = time.time()
        r = parse_demo(d)
        DEM_ROUNDS += r
        if r:
            tw = sum(x["winner"] == "T" for x in r)
            print(f"   {d.name:<40} {len(r):>3} rounds   T {tw} - {len(r) - tw} CT"
                  f"   {sum(x['plant_tick'] is not None for x in r)} plants   [{time.time() - t:.0f}s]")
print(f"total: {len(DEM_ROUNDS)} rounds")

## 4. Mapping world units onto the radar crop

Two steps. The map's overview file turns world units into `[0, 1]` on the 1024 px overview
image. Then a small fit (one scale, two offsets) lines that up with **your** crop, by
maximising how many demo positions land on walkable radar pixels while covering as much of the
walkable area as possible. The second term stops the fit from shrinking all positions into the
middle of the map, which would also score 100% on-map.

Measured on `vitality_vs_falcons`: the overview alone puts **99.1%** of positions on walkable
pixels, and the fit nudges that by about 2 px. **Check the overlay by eye:** green should trace
the corridors. If the crop box ever changes, delete `radar_fit.json` and re-run this cell.

In [ ]:
def world_to_radar(world: np.ndarray, fit: Dict[str, float]) -> np.ndarray:
    """(..., 2) world X,Y -> (..., 2) crop-normalised x right, y down. NaN stays NaN."""
    u = (world[..., 0] - CFG.overview_pos_x) / CFG.overview_scale / CFG.overview_px
    v = (CFG.overview_pos_y - world[..., 1]) / CFG.overview_scale / CFG.overview_px
    return np.stack([u * fit["s"] + fit["ox"], v * fit["s"] + fit["oy"]], -1).astype(np.float32)


def _fit_score(uv: np.ndarray, walk: np.ndarray) -> Tuple[float, float]:
    H, W = walk.shape
    x, y = (uv[:, 0] * W).astype(int), (uv[:, 1] * H).astype(int)
    inside = (x >= 0) & (x < W) & (y >= 0) & (y < H)
    on_map = walk[y[inside], x[inside]].mean() * inside.mean() if inside.any() else 0.0
    occ = np.zeros_like(walk, np.uint8)
    occ[y[inside], x[inside]] = 1
    occ = cv2.dilate(occ, np.ones((5, 5), np.uint8)).astype(bool)
    return float(on_map), float((occ & walk).sum() / max(walk.sum(), 1))


def fit_radar(rounds: List[Dict], bg: np.ndarray, dark: int = 25) -> Dict[str, float]:
    pts = np.concatenate([r["world"].reshape(-1, 2) for r in rounds])
    pts = pts[np.isfinite(pts).all(1)]
    walk = bg.mean(-1) > dark                                  # radar is black off-map
    ident = {"s": 1.0, "ox": 0.0, "oy": 0.0}
    best, best_val = ident, -1.0
    for s in np.arange(0.90, 1.101, 0.01):
        for ox in np.arange(-0.06, 0.0601, 0.005):
            for oy in np.arange(-0.06, 0.0601, 0.005):
                f = {"s": float(s), "ox": float(ox), "oy": float(oy)}
                on, cov = _fit_score(world_to_radar(pts, f), walk)
                if on * cov > best_val:
                    best, best_val = f, on * cov
    on0, cov0 = _fit_score(world_to_radar(pts, ident), walk)
    on1, cov1 = _fit_score(world_to_radar(pts, best), walk)
    print(f"overview only : {on0 * 100:5.1f}% on map, {cov0 * 100:5.1f}% of map covered")
    print(f"fitted        : {on1 * 100:5.1f}% on map, {cov1 * 100:5.1f}% of map covered   {best}")
    return best


def show_fit(rounds: List[Dict], bg: Optional[np.ndarray], fit: Dict[str, float]):
    bg = PLOT_BG if bg is None else bg
    pts = np.concatenate([world_to_radar(r["world"], fit).reshape(-1, 2) for r in rounds])
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    if bg is not None:
        ax.imshow(bg, extent=[0, 1, 1, 0])
    ax.scatter(pts[:, 0], pts[:, 1], s=0.4, c="lime", alpha=0.35)
    for name, (x, y) in SITE_ANCHORS.items():
        ax.text(x, y, name, color="white", fontsize=14, ha="center", va="center", weight="bold")
    ax.set_xlim(0, 1); ax.set_ylim(1, 0); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"demo positions on your radar (n={len(pts)})"); plt.show()


fit_path = Path(CFG.radar_fit_json)
if fit_path.exists():
    RADAR_FIT = json.loads(fit_path.read_text())
    print("loaded", fit_path, RADAR_FIT)
elif DEM_ROUNDS and BG is not None:
    RADAR_FIT = fit_radar(DEM_ROUNDS, BG)
    fit_path.write_text(json.dumps(RADAR_FIT, indent=1))
else:
    RADAR_FIT = {"s": 1.0, "ox": 0.0, "oy": 0.0}
    print("no footage to fit against — using the overview mapping alone")

for r in DEM_ROUNDS:
    r["positions"] = world_to_radar(r["world"], RADAR_FIT)
    if r["plant_world"] is not None:
        pxy = world_to_radar(np.array(r["plant_world"], np.float32), RADAR_FIT)
        r["plant_xy"] = tuple(float(v) for v in pxy)
        r["plant_site"] = min(SITE_ANCHORS, key=lambda k: np.hypot(pxy[0] - SITE_ANCHORS[k][0],
                                                                   pxy[1] - SITE_ANCHORS[k][1]))
    else:
        r["plant_xy"], r["plant_site"] = None, None

if DEM_ROUNDS:
    show_fit(DEM_ROUNDS, PLOT_BG, RADAR_FIT)

## 5. Looking at rounds

Each panel is one round: **orange = T, blue = CT**, a dot where each path starts, **×** where
the bomb went down. Trails end where a player died. The overview grid is the quickest way to
sanity-check a demo, and it is also a fast way to label: most rushes are obvious at a glance.

In [ ]:
def show_round(r: Dict, ax=None, bg: Optional[np.ndarray] = None, label: Optional[str] = None):
    own = ax is None
    if own:
        fig, ax = plt.subplots(figsize=(5, 5))
    bg = PLOT_BG if bg is None else bg
    if bg is not None:
        ax.imshow(bg, extent=[0, 1, 1, 0])
    else:
        ax.set_facecolor("#0c1018")
    for team, sl in (("T", slice(0, CFG.n_per_team)), ("CT", slice(CFG.n_per_team, None))):
        for p in r["positions"][:, sl].transpose(1, 0, 2):
            p = p[np.isfinite(p).all(1)]
            if len(p) < 2:
                continue
            ax.plot(p[:, 0], p[:, 1], color=TEAM_COLOUR[team], lw=1.1, alpha=0.8)
            ax.plot(p[0, 0], p[0, 1], "o", color=TEAM_COLOUR[team], ms=2.5)
    if r.get("plant_xy"):
        ax.plot(*r["plant_xy"], "x", color="red", ms=9, mew=2.2)
    lab = label if label is not None else r.get("label")
    ax.set_title(f"R{r['round_idx']}  {r['duration_s']:.0f}s  win {r['winner']}"
                 f"  plant {r['plant_site'] or '-'}" + (f"\n{lab}" if lab else ""), fontsize=8)
    ax.set_xlim(0, 1); ax.set_ylim(1, 0); ax.set_xticks([]); ax.set_yticks([])
    if own:
        plt.show()


def show_match(demo: str, cols: int = 6, labels: Optional[Dict[int, str]] = None):
    rs = [r for r in DEM_ROUNDS if r["demo"] == demo]
    rows = int(np.ceil(len(rs) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(2.6 * cols, 2.8 * rows), squeeze=False,
                             layout="constrained")
    for ax in axes.ravel():
        ax.axis("off")
    for ax, r in zip(axes.ravel(), rs):
        ax.axis("on")
        show_round(r, ax, label=(labels or {}).get(r["round_idx"]))
    fig.suptitle(demo); plt.show()


if DEM_ROUNDS:
    show_match(DEM_ROUNDS[0]["demo"])

## 6. Labelling demo rounds

The winner, round boundaries and plant site already come from the demo, so labelling is one
keypress per round:

```
  1-5          A_RUSH  B_RUSH  A_SPLIT  B_SPLIT  RETAKE
  x            exclude: fits no class (slow default, eco, ...)
  backspace    clear the label
  left / right previous / next round
  s / q        save / save and quit
```

**`x` is deliberate.** The five classes have no bucket for a slow default round (CLAUDE.md §9).
Excluding instead of forcing keeps every class clean, and the count of excluded rounds becomes
the evidence for deciding whether a `DEFAULT` class is worth adding.

Needs a live window: run the notebook with `%matplotlib widget` (`pip install ipympl`), or from
a terminal. With `%matplotlib inline` the keys go nowhere.

Labels are saved per demo to `labels/<demo>.demo_rounds.json`, keyed by round number, and
the labeller resumes where you left off.

In [ ]:
def demo_labels_path(demo: str) -> Path:
    return Path(CFG.labels_dir) / f"{demo}.demo_rounds.json"


def load_demo_labels(demo: str) -> Dict[int, str]:
    p = demo_labels_path(demo)
    return {int(k): v for k, v in json.loads(p.read_text())["rounds"].items()} if p.exists() else {}


class DemoRoundLabeller:
    KEYS = {str(i + 1): c for i, c in enumerate(CLASSES)} | {"x": EXCLUDED}

    def __init__(self, demo: str):
        self.demo = demo
        self.rounds = [r for r in DEM_ROUNDS if r["demo"] == demo]
        if not self.rounds:
            raise ValueError(f"no parsed rounds for {demo}")
        self.labels = load_demo_labels(demo)
        unl = [i for i, r in enumerate(self.rounds) if r["round_idx"] not in self.labels]
        self.i = unl[0] if unl else 0                         # resume at the first unlabelled
        print(f"{demo}: {len(self.rounds)} rounds, {len(self.labels)} already labelled")

    def save(self):
        p = demo_labels_path(self.demo)
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(json.dumps({"tactics": CLASSES, "rounds": {str(k): v for k, v in
                                                                sorted(self.labels.items())}},
                                indent=1))
        counts = {c: sum(v == c for v in self.labels.values()) for c in CLASSES + [EXCLUDED]}
        print(f"saved {len(self.labels)}/{len(self.rounds)} -> {p}   {counts}")

    def _draw(self):
        r = self.rounds[self.i]
        self.ax.clear()
        show_round(r, self.ax, label=self.labels.get(r["round_idx"], "— unlabelled —"))
        self.ax.set_xlabel("1 A_RUSH  2 B_RUSH  3 A_SPLIT  4 B_SPLIT  5 RETAKE  x exclude\n"
                           f"round {self.i + 1}/{len(self.rounds)}   "
                           f"labelled {len(self.labels)}", fontsize=8)
        self.fig.canvas.draw_idle()

    def _key(self, ev):
        rid = self.rounds[self.i]["round_idx"]
        if ev.key in self.KEYS:
            self.labels[rid] = self.KEYS[ev.key]
            self.i = min(self.i + 1, len(self.rounds) - 1)    # advance after labelling
        elif ev.key == "backspace":
            self.labels.pop(rid, None)
        elif ev.key == "right":
            self.i = min(self.i + 1, len(self.rounds) - 1)
        elif ev.key == "left":
            self.i = max(self.i - 1, 0)
        elif ev.key == "s":
            self.save()
        elif ev.key == "q":
            self.save(); plt.close(self.fig); return
        self._draw()

    def run(self):
        self.fig, self.ax = plt.subplots(figsize=(6.5, 7))
        self.fig.canvas.mpl_connect("key_press_event", self._key)
        self._draw(); plt.show()


print("label a demo:   DemoRoundLabeller(DEMOS[k].stem).run()")
for d in sorted({r["demo"] for r in DEM_ROUNDS}):
    n = sum(r["demo"] == d for r in DEM_ROUNDS)
    print(f"   {d:<40} {len(load_demo_labels(d)):>3}/{n} labelled")

## 7. Export for the main notebook

Writes every **labelled, non-excluded** round to `dem_rounds.npz` in the main notebook's
format. To train Model 2 on them there:

```python
d = np.load("dem_rounds.npz", allow_pickle=True)
dem_rounds = [Round(str(m), int(i), str(l), p, None, str(w))
              for m, i, l, p, w in zip(d["match_ids"], d["round_idx"], d["labels"],
                                       d["positions"], d["winners"])]
```

**Splitting rule: by `match_groups`, not `match_ids`.** A demo and its rendered video are the
same match. If the demo trains Model 2 and the video tests it, that is the "revising with the
exam paper" leak from CLAUDE.md §5. The pairing table below shows which demos have a video.

In [ ]:
def pairing_table() -> None:
    videos = sorted(p for d in (CFG.data_dir, CFG.radar_dir) for p in Path(d).glob("*.mp4")
                    if not p.name.startswith(("~", ".")))
    vid_groups: Dict[str, List[str]] = {}
    for v in videos:
        vid_groups.setdefault(match_group(v.stem), []).append(f"{Path(v.parent.name) / v.name}")
    print(f"{'demo':<40} {'match_group':<24} rendered video")
    for d in sorted({r["demo"] for r in DEM_ROUNDS}):
        g = match_group(d)
        print(f"{d:<40} {g:<24} {', '.join(vid_groups.get(g, [])) or '-'}")
    orphans = [g for g in vid_groups if g not in {r["match_group"] for r in DEM_ROUNDS}]
    if orphans:
        print("videos with no demo:", orphans)


def build_dem_dataset(out_npz: str = None) -> int:
    rows = []
    for r in DEM_ROUNDS:
        lab = load_demo_labels(r["demo"]).get(r["round_idx"])
        if lab in CLASSES:                                    # skips unlabelled and EXCLUDED
            rows.append((r, lab))
    if not rows:
        print("no labelled rounds yet — use DemoRoundLabeller"); return 0
    rs, labs = zip(*rows)
    np.savez_compressed(
        out_npz or CFG.dem_npz,
        positions=np.stack([r["positions"] for r in rs]),     # (R, 32, 10, 2) crop-normalised
        world=np.stack([r["world"] for r in rs]),             # raw world units, for re-fitting
        labels=np.array(labs), winners=np.array([r["winner"] for r in rs]),
        match_ids=np.array([r["demo"] for r in rs]),
        match_groups=np.array([r["match_group"] for r in rs]),
        round_idx=np.array([r["round_idx"] for r in rs]),
        plant_site=np.array([r["plant_site"] or "" for r in rs]),
        duration_s=np.array([r["duration_s"] for r in rs], np.float32))
    print(f"{len(rs)} rounds from {len({r['demo'] for r in rs})} demo(s) -> {out_npz or CFG.dem_npz}")
    print("per class:", {c: labs.count(c) for c in CLASSES})
    return len(rs)


pairing_table()
build_dem_dataset()

## 8. What this does not do yet

* **Sync a demo to its video.** Needed for the strongest evaluation: tracker positions vs.
  demo truth on the *same* frames, which measures the detector + tracker error on real footage
  without hand labels, and can also produce icon labels for the detector automatically. Doable
  by matching the radar's first moving frame after each `round_freeze_end`, but not built.
* **Wire `dem_rounds.npz` into the main notebook's Experiment A.** Deliberately left for you
  to review first; the snippet in §7 is all it takes.